# Filtro de quebradas para temporadas El Niño

Qué quebradas de Lurigancho-Chosica, Chaclacayo y el valle de Santa Eulalia tienen más probabilidad de activarse como huaico en una temporada con El Niño.

> **No es un sistema de alerta ni una predicción para decidir evacuaciones.** Es un ranking de investigación, con la evidencia que hay y sus huecos a la vista. Ante una emergencia llama al 115 (INDECI).

Antes de correrlo: `uv run descargar-datos --solo copernicus-dem osm-aguas osm-edificios noaa-enso` y `uv run python -m illapa_research.quebradas` (delimita las cuencas, ~30 s).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from shapely.geometry import shape

from illapa_research import filtro, nino
from illapa_research.catalogo import CATALOGO, cargar_cuencas, cargar_todos
from illapa_research.quebradas import cargar_referencias

DATOS = Path("../datos") if Path("../datos").exists() else Path("datos")
cuencas = filtro.cuencas_a_tabla(cargar_cuencas())
print(len(cuencas), "cuencas de 1 km² o más que drenan al Rímac o a Santa Eulalia")

## 1. Cuencas y comprobación contra valores publicados

Las cuencas salen del DEM Copernicus GLO-30 (30 m) con el cauce del Rímac y de Santa Eulalia (OSM) hundido 30 m para que el flujo siga el río real. Cada cuenca llega hasta su confluencia con el río. Referencias: IGP 2020 (Quirio 10,3 km² según el cuadro 01; otras fuentes dan 9 a 11; Chacrasana 4,7, La Ronda 8,9, Barba Blanca 3,5, ubicadas con sus coordenadas UTM) y el piloto previo (Pedregal 9,6, Los Laureles 8,95).

In [ ]:
nombradas = cuencas[cuencas.ubicacion == "referencia"][["nombre", "area_km2", "relieve_m", "pendiente_media_pct", "longitud_cauce_km", "melton"]]
nombradas.round(2)

In [ ]:
# Cuencas con edificios cerca de la desembocadura y sin nombre ubicado
cuencas[(cuencas.ubicacion == "sin_nombre") & (cuencas.edificios_1km > 500)][["slug", "salida_lon", "salida_lat", "area_km2", "edificios_1km"]].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for f in cargar_cuencas()["features"]:
    g = shape(f["geometry"])
    p = f["properties"]
    ax.fill(*g.exterior.xy if g.geom_type == "Polygon" else g.convex_hull.exterior.xy,
            alpha=0.5, color="tab:red" if p["ubicacion"] == "referencia" else "tab:blue", lw=0.5)
ax.set_aspect(1.0); ax.set_title("Cuencas delimitadas (rojo: con nombre ubicado)")
plt.show()

## 2. Historial de activaciones por quebrada

El campo `quebradas` del catálogo es texto libre. `catalogo/quebradas_referencias.csv` dice a qué cuenca corresponde cada nombre, **solo cuando hay un punto de referencia verificable**. Lo demás queda como «nombre sin ubicar».

In [ ]:
eventos = cargar_todos(solo_dia_exacto=True, sin_baja_confianza=True)
refs = cargar_referencias(CATALOGO / "quebradas_referencias.csv")
hist = filtro.asignar_historial(eventos, refs, cuencas)
print(hist.resumen)
pd.Series(hist.sin_ubicar, name="fechas").head(15)

## 3. Temporadas con El Niño

Temporada = diciembre a abril, con el año de enero. Niño global: ONI (Niño 3.4) medio de DJF, JFM y FMA ≥ +0,5 °C. Niño costero: anomalía media de Niño 1+2 de enero a abril ≥ +1,0 °C (aproximación; el índice oficial es el ICEN del ENFEN/IGP, que aún no se descarga). Los umbrales se fijaron antes de mirar los huaicos. Fuente: NOAA CPC.

In [ ]:
temporadas = nino.temporadas(nino.cargar_oni(DATOS / "enso" / "noaa_oni.txt"), nino.cargar_nino12(DATOS / "enso" / "noaa_nino12.txt"))
temporadas[temporadas.nino].round(2)

In [ ]:
filtro.efecto_nino(eventos, temporadas)

## 4. Ranking

Nivel 1 y 2: quebradas con activaciones documentadas en temporadas Niño. Niveles 3 a 5: sin historial ubicado, ordenadas por susceptibilidad física (media de los percentiles de Melton, pendiente del cauce y pendiente media entre cuencas de 2 a 50 km²). Nivel 6: menos de 2 km², no evaluable con un DEM de 30 m. **Un nivel bajo no quiere decir «no se activa»**: puede ser que el nombre no se haya podido ubicar.

In [ ]:
ranking = filtro.tabla(cuencas, hist, temporadas)
ranking[["posicion", "slug", "nombre", "area_km2", "fisica", "edificios_1km", "temporadas_nino_activas", "temporadas_nino", "tasa_nino", "tasa_nino_ic80_bajo", "tasa_nino_ic80_alto", "nivel"]].head(25).round(2)

## 5. Validación dejando un Niño fuera

Se quita una temporada Niño, se rehace el ranking con las demás y se mira dónde queda lo que sí se activó. Solo pueden aparecer las quebradas ubicadas, que son las más conocidas: esto comprueba consistencia, no acierto en quebradas que nadie ha registrado.

In [ ]:
loo = filtro.dejar_un_nino_fuera(cuencas, hist, temporadas)
print(loo.groupby("slug")[["percentil_fisica", "con_historial"]].mean().round(1))
loo.groupby("slug").nivel.value_counts().unstack(fill_value=0)

## Qué NO se puede concluir

- La lluvia y el Niño explican poco por sí solos: ver `02-estudio-de-eventos`.
- Solo 43 de 147 fechas se pudieron ubicar en una cuenca; 15 cuencas tienen nombre ubicado. Los «aciertos» del historial son de las quebradas más famosas.
- El registro de prensa y de instituciones tiene huecos (1999 a 2008) y más cobertura en años de desastre.
- La susceptibilidad física **no** discrimina bien entre las quebradas con historial: acierta con Barba Blanca (percentil 14) pero deja a casi todas las demás en el tramo medio o bajo (39 a 82). Se reporta tal cual, sin reajustar los pesos.
- Con un modelo de 30 m las cuencas de menos de 2 km² no son fiables.